# Practice Lab: LoRA, QLoRA & Adapters (Lesson 9.5)

Module 9 · 8 exercises · Rank ablation + target modules + variants + quantization + LR sweep

Exercises 1, 2, 3, 4, 8 run locally (pure Python).


# Lesson 9.5: LoRA, QLoRA & Adapters

8 exercises: 2E + 3M + 3C

Exercises 1, 2, 3, 4, 8 run **locally** (pure Python). Ex 5, 6, 7 are design.


In [ ]:
import numpy as np
np.random.seed(42)


---
## Exercise 1 (Easy): Rank Ablation Experiment


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import numpy as np
np.random.seed(42)

h, L, n_mod = 4096, 32, 7
ranks = [4, 8, 16, 32, 64, 128]

print("Rank Ablation (fixed LR=2e-4):")
print(f"{'Rank':<8} {'Params':>12} {'%':>7} {'Size':>8} {'Loss':>8}")
print("-"*48)
for r in ranks:
    p = 2*h*r*n_mod*L; pct = p/6.7e9*100; sz = p*2/1e6
    loss = 1.5 - np.log2(r+1)*0.08 + np.random.normal(0,0.02)
    print(f"  r={r:<5} {p:>12,} {pct:>6.2f}% {sz:>6.0f}MB {loss:>8.3f}")

print(f"\nDiminishing returns after r=16")
print(f"Fix: rsLoRA (alpha/sqrt(r)) or sweep LR per rank")
print(f"Hierarchy: LR tuning > target modules > rank")


</details>


---
## Exercise 2 (Easy): Target Module Ablation


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import numpy as np
np.random.seed(42)

configs = [
    ("q+v only (classic)", 2, 0.70),
    ("All attention (q/k/v/o)", 4, 0.82),
    ("MLP-only (gate/up/down)", 3, 0.85),
    ("All-linear (7 modules)", 7, 0.95),
]

print("Target Module Ablation (r=16, Llama 7B):")
print(f"{'Config':<30} {'Mods':>6} {'Params':>12} {'Quality':>9}")
print("-"*60)
for name, n, qf in configs:
    p = 2*4096*16*n*32; q = qf + np.random.normal(0,0.02)
    best = " <--" if n==7 else ""
    print(f"  {name:<28} {n:>6} {p:>12,} {q:>8.3f}{best}")

print(f"\nMLP-only > attention-only (Schulman 2025)")
print(f"All-linear recovers full FT quality (QLoRA paper)")
print(f"Rule: target_modules='all-linear'")


</details>


---
## Exercise 3 (Medium): DoRA vs rsLoRA vs Vanilla


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import numpy as np
np.random.seed(42)

tasks = ["Classification", "Generation", "Math"]
base = [0.85, 0.78, 0.62]
variants = [
    ("Vanilla LoRA", [0,0,0], "default"),
    ("+DoRA", [0.02,0.03,0.04], "use_dora=True"),
    ("+rsLoRA", [0.01,0.01,0.02], "use_rslora=True"),
    ("+DoRA+rsLoRA", [0.03,0.04,0.05], "use_dora=True, use_rslora=True"),
    ("+PiSSA", [0.02,0.03,0.05], 'init_lora_weights="pissa"'),
]

print("LoRA Variants Benchmark (r=16):")
print(f"{'Variant':<18}", end="")
for t in tasks: print(f" {t:>14}", end="")
print(f" {'Avg':>8}")
print("-"*62)
for name, boost, flag in variants:
    scores = [round(base[i]+boost[i]+np.random.normal(0,0.005),3) for i in range(3)]
    print(f"  {name:<16}", end="")
    for s in scores: print(f" {s:>14.3f}", end="")
    print(f" {np.mean(scores):>8.3f}")

print(f"\nFlags: use_dora=True, use_rslora=True (production default)")
print(f"DoRA: +1-4%, zero inference overhead, beat full FT on math")
print(f"PiSSA: +5% GSM8K but FAILS in GRPO (never for RL)")


</details>


---
## Exercise 4 (Medium): NF4 vs FP4 vs INT4


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import numpy as np
np.random.seed(42)

weights = np.random.normal(0, 0.02, 100000)
w_range = np.max(np.abs(weights))

# INT4: uniform
int4 = np.linspace(-w_range, w_range, 16)
# FP4: log spacing
fp4 = np.concatenate([-np.logspace(-4, np.log10(w_range), 8)[::-1],
                       np.logspace(-4, np.log10(w_range), 8)])
# NF4: quantile-based
quantiles = np.linspace(0, 1, 17)
nf4_q = np.quantile(weights, quantiles[:-1])
nf4 = (nf4_q[:-1] + nf4_q[1:]) / 2

def qerr(w, levels):
    idx = np.argmin(np.abs(w[:,None] - levels[None,:]), axis=1)
    return np.mean((w - levels[idx])**2)

results = {"INT4 (uniform)": qerr(weights, int4),
           "FP4 (log)": qerr(weights, fp4),
           "NF4 (quantile)": qerr(weights, nf4)}

print("Quantization Comparison (100K weights, N(0,0.02)):")
best_name = min(results, key=results.get)
for name, mse in sorted(results.items(), key=lambda x: x[1]):
    red = (1 - mse/max(results.values()))*100
    win = " <-- BEST" if name == best_name else ""
    print(f"  {name:<20} MSE: {mse:.2e} ({red:.0f}% less error){win}")

near_zero = np.sum(np.abs(weights) < 0.02) / len(weights)
print(f"\n{near_zero:.0%} of weights near zero -> NF4 allocates more precision there")
print(f"Double quant saves: 7B={7*0.37/8:.1f}GB, 65B={65*0.37/8:.1f}GB")
print(f"Always: bnb_4bit_use_double_quant=True")


</details>


---
## Exercise 8 (Challenge): LR Sweep Experiment


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import numpy as np
np.random.seed(42)

def sweep(rank, lrs):
    opt_lr = 2e-4 / np.sqrt(rank/16)
    return [{"lr":lr, "acc": round(np.clip(0.90-0.15*(np.log10(lr/opt_lr))**2+np.random.normal(0,0.005),0.5,0.95),4)} for lr in lrs]

lrs = [1e-5, 3e-5, 1e-4, 2e-4, 5e-4, 1e-3, 2e-3, 5e-3]
r16 = sweep(16, lrs); r128 = sweep(128, lrs)

print("LR Sweep Experiment:")
print(f"  {'LR':<10} {'r=16':>10} {'r=128':>10}")
print("-"*34)
for a, b in zip(r16, r128):
    print(f"  {a['lr']:<10.0e} {a['acc']:>10.4f} {b['acc']:>10.4f}")

b16 = max(r16, key=lambda x:x["acc"]); b128 = max(r128, key=lambda x:x["acc"])
print(f"\nBest: r=16 LR={b16['lr']:.0e} acc={b16['acc']}")
print(f"      r=128 LR={b128['lr']:.0e} acc={b128['acc']}")
print(f"  Diff when LR tuned: {abs(b16['acc']-b128['acc']):.4f} (<0.43%)")

# Wrong LR cost
wrong = next(r for r in r128 if r["lr"]==b16["lr"])
print(f"  r=128 with r=16's LR: {wrong['acc']:.4f} (mismatch cost: {b128['acc']-wrong['acc']:.4f})")
print(f"\nProved: LR tuning > rank selection > variant selection")


</details>


---
## Exercise 5 (Medium): Adapter Merging
Design/architecture. See practice-lab-9_5.html.


In [ ]:
# Exercise 5: Adapter Merging
pass


---
## Exercise 6 (Challenge): Multi-LoRA Serving
Design/architecture. See practice-lab-9_5.html.


In [ ]:
# Exercise 6: Multi-LoRA Serving
pass


---
## Exercise 7 (Challenge): Indic Multilingual Adapters
Design/architecture. See practice-lab-9_5.html.


In [ ]:
# Exercise 7: Indic Multilingual Adapters
pass
